# WSP Toolkit — M3 demo on Kaggle: Qwen3.5-0.8B base vs. the WSP edit

Compares the base model with the authors' edited model (the "loop fix") under the paper's sampling settings, and runs
three non-edit baselines. Results are written to `/kaggle/working/` (download them from the **Output** tab).

**Before running:** in the right-hand panel set **Accelerator → GPU T4 x2** and **Internet → On** (Internet needs a
phone-verified Kaggle account). Then use **Save Version → Save & Run All (Commit)** so it runs in the background (up to
12 h) without your browser open.

Settings are in the next code cell. The notebook first measures real throughput, then shrinks `LIMIT` if needed so the
whole run fits in `TIME_BUDGET_HOURS`.

In [ ]:
# ---- settings ---------------------------------------------------------------
LIMIT = 30                  # MATH-500 problems per run (the paper used all 500 x 4 samples; that needs much more GPU)
MAX_NEW_TOKENS = 32768      # paper setting; the base model hits this on ~93% of problems in the paper
SUITES = "math500_heldout"  # add ",gsm8k_heldout" for GSM8K too (roughly doubles the time)
BATCH = 8                   # problems generated together per GPU
DTYPE = "float32"           # T4/P100 have no bfloat16; float32 is the safe choice (float16 may overflow in Qwen)
TIME_BUDGET_HOURS = 9.0     # leave margin under Kaggle's 12 h limit, or outputs are lost
REPO = "https://github.com/encore488/weight-space-manipulator.git"
BRANCH = "main"


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; N_GPU = torch.cuda.device_count(); print("GPUs:", N_GPU)
assert N_GPU >= 1, "No GPU: set Accelerator to 'GPU T4 x2' in the notebook settings"


In [ ]:
%cd /kaggle/working
!rm -rf weight-space-manipulator && git clone -q -b $BRANCH $REPO
%cd /kaggle/working/weight-space-manipulator
!pip install -q -U "transformers>=5" huggingface_hub pyarrow 2>&1 | tail -2
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q flash-linear-attention 2>&1 | tail -1 || echo "optional fast kernels not installed (fine, just slower)"
import transformers; print("transformers", transformers.__version__)


In [ ]:
# download both checkpoints once (about 1.6 GB each) so the two GPU processes don't race
from huggingface_hub import snapshot_download
pat = ["*.json", "*.safetensors", "*.txt", "*.jinja", "tokenizer*"]
snapshot_download("Qwen/Qwen3.5-0.8B", revision="2fc06364715b967f1860aea9cf38778875588b17", allow_patterns=pat)
snapshot_download("RikiyaT/Qwen3.5-0.8B-weight-space-programming", allow_patterns=pat)
print("downloaded")


In [ ]:
# measure real throughput: one batch, 512 new tokens, on GPU 0
import os, json, subprocess, math
W = "/kaggle/working"
env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "WSP_DATA_DIR": f"{W}/wsp_smoke"}
subprocess.run(["python", "scripts/qwen_m3_demo.py", "--confirm-download", "--device", "cuda", "--dtype", DTYPE,
                "--batch-size", str(BATCH), "--smoke", "--runs", "base", "--suites", "math500_heldout",
                "--results-jsonl", f"{W}/smoke.jsonl"], env=env, check=True)
tps = json.loads(open(f"{W}/smoke.jsonl").read().splitlines()[-1])["tokens_per_second"]
# long contexts are slower than this short test, so assume half the speed; worst case every problem hits the limit
runs_per_gpu = 5 if N_GPU == 1 else 3
fit = int(TIME_BUDGET_HOURS * 3600 * tps * 0.5 / (runs_per_gpu * MAX_NEW_TOKENS * len(SUITES.split(","))))
print(f"measured {tps} tokens/s -> at most {fit} problems per run fit in {TIME_BUDGET_HOURS} h (worst case)")
if fit < LIMIT:
    LIMIT = max(5, fit)
    print("reducing LIMIT to", LIMIT)
print("LIMIT =", LIMIT)


In [ ]:
# the real run: split across both GPUs (or run everything on one)
import time
common = ["python", "scripts/qwen_m3_demo.py", "--confirm-download", "--device", "cuda", "--dtype", DTYPE,
          "--batch-size", str(BATCH), "--limit", str(LIMIT), "--max-new-tokens", str(MAX_NEW_TOKENS),
          "--suites", SUITES]
plans = ([("0", "base,baseline:repetition_penalty_1.1,baseline:presence_penalty_2.0"),
          ("1", "edited,baseline:temperature_1.0")] if N_GPU >= 2 else
         [("0", "base,edited,baseline:repetition_penalty_1.1,baseline:presence_penalty_2.0,baseline:temperature_1.0")])
procs = []
for gpu, runs in plans:
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": gpu, "WSP_DATA_DIR": f"{W}/wsp_gpu{gpu}"}
    log = open(f"{W}/log_gpu{gpu}.txt", "w")
    procs.append((gpu, subprocess.Popen(common + ["--runs", runs, "--results-jsonl", f"{W}/results_gpu{gpu}.jsonl"],
                                        env=env, stdout=log, stderr=subprocess.STDOUT)))
start = time.time()
while any(p.poll() is None for _, p in procs):
    time.sleep(600)
    print(f"--- {(time.time() - start) / 3600:.1f} h elapsed")
    for gpu, _ in procs:
        lines = [l for l in open(f"{W}/log_gpu{gpu}.txt").read().splitlines() if "problems" in l or '"suite"' in l]
        print(f"GPU {gpu}:", lines[-1] if lines else "(loading)")
for gpu, p in procs:
    print(f"GPU {gpu} exit code {p.returncode}")
    print(open(f"{W}/log_gpu{gpu}.txt").read()[-3000:])


In [ ]:
# merge results + weight diff into one report
import glob
files = sorted(glob.glob(f"{W}/results_gpu*.jsonl"))
subprocess.run(["python", "scripts/qwen_m3_demo.py", "--report-from", *files, "--with-diff",
                "--out", f"{W}/m3_qwen_demo.md"], env={**os.environ, "WSP_DATA_DIR": f"{W}/wsp_gpu0",
                                                      "CUDA_VISIBLE_DEVICES": ""}, check=True)


## Send the results back

From the notebook's **Output** tab download `m3_qwen_demo.md` and the `results_gpu*.jsonl` files (and `log_gpu*.txt`
if anything failed), then upload them to the Claude Code session. They'll be committed to `docs/reports/`.